# 📊 Smart Library — Dashboard des vraies données
## Notebook 07 : Visualisation des données depuis MinIO/Iceberg

---
Ce notebook lit les vraies données depuis MinIO (Iceberg, via le catalogue REST
Lakekeeper) et génère un dashboard HTML interactif avec les résultats réels.

### Ce qu'on visualise
- **Bronze** : structure et volume des données brutes
- **Silver** : qualité des données après nettoyage
- **Gold** : analytics business (ventes, RFM, tendances)
- **Streaming** : événements temps réel
- **Recommandations** : résultats KNN + ALS

---
## Étape 0 — Initialisation

In [1]:
import os, sys, time, re
from datetime import datetime
from pathlib import Path

# ── Tuer toute session existante ─────────────────────────────
try:
    from pyspark.sql import SparkSession
    SparkSession.getActiveSession().stop()
except:
    pass

# ── Java ──────────────────────────────────────────────────────
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"]      = os.environ["JAVA_HOME"] + "/bin:" + os.environ.get("PATH", "")

# ── Jars Iceberg + MinIO ──────────────────────────────────────
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages "
    "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.5.2,"
    "org.apache.iceberg:iceberg-aws-bundle:1.5.2,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262 "
    "pyspark-shell"
)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Connexion MinIO + Lakekeeper ──────────────────────────────
MINIO_ENDPOINT       = "http://minio:9000"
MINIO_ACCESS         = "minioadmin"
MINIO_SECRET         = "minioadmin123"
CATALOG_NAME         = "lakehouse"
LAKEKEEPER_URI       = "http://lakekeeper:8181/catalog"
LAKEKEEPER_WAREHOUSE = "lakehouse"

NS_BRONZE = f"{CATALOG_NAME}.bronze"
NS_SILVER = f"{CATALOG_NAME}.silver"
NS_GOLD   = f"{CATALOG_NAME}.gold"

# ── Fonctions utilitaires Iceberg (équivalent hudi_options) ───
#
#  Avec Hudi : le type de table (COW/MOR) et la clé d'upsert se
#  déclarent au niveau de l'ÉCRITURE (options du DataFrameWriter).
#  Avec Iceberg : ce sont des PROPRIÉTÉS DE TABLE, et l'UPSERT
#  s'exprime explicitement en SQL via MERGE INTO.
#
def iceberg_table_properties(type_table="copy-on-write"):
    """
    type_table :
      - "copy-on-write" → équivalent Hudi COW
      - "merge-on-read"  → équivalent Hudi MOR
    """
    return {
        "format-version"    : "2",
        "write.delete.mode" : type_table,
        "write.update.mode" : type_table,
        "write.merge.mode"  : type_table,
    }

def create_or_replace_table(df, table_name, partition_cols=None, type_table="copy-on-write"):
    """Équivalent de : df.write.format('hudi')....mode('overwrite').save(chemin)"""
    # Iceberg (ClusteredWriter) exige que les lignes soient triées/regroupées
    # par colonne de partition AVANT l'écriture — contrairement à Hudi qui
    # gère le clustering en interne. Sans ce tri : "Incoming records violate
    # the writer assumption that records are clustered by spec and partition".
    if partition_cols:
        df = df.sort(*partition_cols)
    writer = df.writeTo(table_name)
    if partition_cols:
        writer = writer.partitionedBy(*[F.col(c) for c in partition_cols])
    for k, v in iceberg_table_properties(type_table).items():
        writer = writer.tableProperty(k, v)
    writer.createOrReplace()

def append_table(df, table_name):
    """Ajout de lignes (insert simple, pas d'upsert)."""
    df.writeTo(table_name).append()

def upsert_table(spark, df, table_name, key_cols):
    """UPSERT via MERGE INTO — équivalent fonctionnel de l'UPSERT automatique Hudi."""
    vue = f"_staging_{table_name.split('.')[-1]}"
    df.createOrReplaceTempView(vue)
    condition = " AND ".join([f"t.{c} = s.{c}" for c in key_cols])
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING {vue} s
        ON {condition}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

def snapshots(spark, table_name):
    """Retourne l'historique des snapshots Iceberg (équivalent de la timeline Hudi)."""
    return spark.sql(f"SELECT * FROM {table_name}.snapshots ORDER BY committed_at")

# ── SparkSession ──────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("SmartLibrary-Dashboard")
    .master("local[2]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.type", "rest")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.uri", LAKEKEEPER_URI)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LAKEKEEPER_WAREHOUSE)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.endpoint",           MINIO_ENDPOINT)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.access-key-id",     MINIO_ACCESS)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.secret-access-key", MINIO_SECRET)
    .config(f"spark.sql.catalog.{CATALOG_NAME}.s3.path-style-access", "true")
    .config("spark.sql.defaultCatalog", CATALOG_NAME)
    .config("spark.serializer",
            "org.apache.spark.serializer.KryoSerializer")
    .config("spark.hadoop.fs.s3a.endpoint",          MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",        MINIO_ACCESS)
    .config("spark.hadoop.fs.s3a.secret.key",        MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl",
            "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# ── Création des namespaces Medallion (idempotent) ────────────
for ns in (NS_BRONZE, NS_SILVER, NS_GOLD):
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {ns}")

print(f"✅ SparkSession démarrée — version {spark.version}")
print(f"   Catalogue Iceberg : {CATALOG_NAME} (REST → Lakekeeper → Postgres)")
print(f"   Stockage          : {MINIO_ENDPOINT} (bucket lakehouse)")

BRONZE_UTILISATEURS = f"{NS_BRONZE}.utilisateurs"
BRONZE_LIVRES        = f"{NS_BRONZE}.livres"
BRONZE_COMMANDES     = f"{NS_BRONZE}.commandes"
BRONZE_STOCK_COW     = f"{NS_BRONZE}.stock_cow"
BRONZE_STOCK_MOR     = f"{NS_BRONZE}.stock_mor"

SILVER_UTILISATEURS = f"{NS_SILVER}.utilisateurs"
SILVER_LIVRES        = f"{NS_SILVER}.livres"
SILVER_COMMANDES     = f"{NS_SILVER}.commandes"
SILVER_STOCK         = f"{NS_SILVER}.stock"

GOLD_TOP_LIVRES     = f"{NS_GOLD}.top_livres"
GOLD_COMPORTEMENT   = f"{NS_GOLD}.comportement_utilisateurs"
GOLD_HISTORIQUE     = f"{NS_GOLD}.historique_ventes"
GOLD_TENDANCES      = f"{NS_GOLD}.tendances"
GOLD_RECOMM         = f"{NS_GOLD}.recommandations"

import json

✅ SparkSession démarrée — version 3.4.0
   Catalogue Iceberg : lakehouse (REST → Lakekeeper → Postgres)
   Stockage          : http://minio:9000 (bucket lakehouse)


---
## Étape 1 — Extraction des vraies données depuis Iceberg

In [2]:
print("📥 Lecture des tables depuis MinIO/Iceberg (catalogue Lakekeeper)...")

# ── Bronze ────────────────────────────────────────────────────
# (pas de colonnes techniques à retirer : les métadonnées Iceberg
#  vivent dans le catalogue, pas dans les lignes de données)
df_b_users     = spark.table(BRONZE_UTILISATEURS)
df_b_livres    = spark.table(BRONZE_LIVRES)
df_b_cmd       = spark.table(BRONZE_COMMANDES)
df_b_stock_cow = spark.table(BRONZE_STOCK_COW)
df_b_stock_mor = spark.table(BRONZE_STOCK_MOR)

# ── Silver ────────────────────────────────────────────────────
df_s_users = spark.table(SILVER_UTILISATEURS)
df_s_livres= spark.table(SILVER_LIVRES)
df_s_cmd   = spark.table(SILVER_COMMANDES)
df_s_stock = spark.table(SILVER_STOCK)

# ── Gold ──────────────────────────────────────────────────────
df_g_top       = spark.table(GOLD_TOP_LIVRES)
df_g_rfm       = spark.table(GOLD_COMPORTEMENT)
df_g_hist      = spark.table(GOLD_HISTORIQUE)
df_g_tendances = spark.table(GOLD_TENDANCES)
df_g_reco      = spark.table(GOLD_RECOMM)

print("✅ Toutes les tables chargées")

📥 Lecture des tables depuis MinIO/Iceberg (catalogue Lakekeeper)...
✅ Toutes les tables chargées


---
## Étape 2 — Calcul des métriques réelles

In [3]:
# ── KPIs globaux ──────────────────────────────────────────────
kpi_commandes   = df_s_cmd.count()
kpi_utilisateurs= df_s_users.count()
kpi_livres      = df_s_livres.count()

kpi_ca = (
    df_s_cmd
    .filter(F.col("statut") == "livré")
    .agg(F.round(F.sum("montant_total"), 2).alias("ca"))
    .first()["ca"] or 0
)

kpi_panier = (
    df_s_cmd
    .filter(F.col("statut") == "livré")
    .agg(F.round(F.avg("montant_total"), 2).alias("pm"))
    .first()["pm"] or 0
)

kpi_taux_livraison = (
    df_s_cmd
    .agg(
        F.round(
            F.sum(F.when(F.col("statut") == "livré", 1).otherwise(0)) * 100.0 /
            F.count("*"), 1
        ).alias("taux")
    ).first()["taux"] or 0
)

print(f"✅ KPIs calculés depuis les vraies données")
print(f"   Commandes          : {kpi_commandes:,}")
print(f"   Utilisateurs       : {kpi_utilisateurs:,}")
print(f"   Livres             : {kpi_livres:,}")
print(f"   CA livré           : {kpi_ca:,} €")
print(f"   Panier moyen       : {kpi_panier} €")
print(f"   Taux livraison     : {kpi_taux_livraison}%")

✅ KPIs calculés depuis les vraies données
   Commandes          : 5,000
   Utilisateurs       : 1,000
   Livres             : 500
   CA livré           : 293,297.83 €
   Panier moyen       : 89.75 €
   Taux livraison     : 65.4%


In [4]:
# ── CA par mois ───────────────────────────────────────────────
ca_mensuel = (
    df_s_cmd
    .filter(F.col("statut") == "livré")
    .groupBy("annee", "mois")
    .agg(F.round(F.sum("montant_total"), 2).alias("ca"),
         F.count("*").alias("nb_cmd"))
    .orderBy("annee", "mois")
    .collect()
)

ca_labels = [f"{r['annee'][-2:]}/{r['mois']}" for r in ca_mensuel]
ca_values = [float(r['ca']) for r in ca_mensuel]
cmd_values= [int(r['nb_cmd']) for r in ca_mensuel]

# ── Ventes par genre ──────────────────────────────────────────
ventes_genre = (
    df_s_cmd
    .filter(F.col("statut").isin(["livré", "en_cours"]))
    .groupBy("genre")
    .agg(F.round(F.sum("montant_total"), 2).alias("ca"),
         F.count("*").alias("nb"))
    .orderBy(F.col("ca").desc())
    .collect()
)

genre_labels = [r['genre'] for r in ventes_genre]
genre_ca     = [float(r['ca']) for r in ventes_genre]
genre_nb     = [int(r['nb']) for r in ventes_genre]

# ── Statuts commandes ─────────────────────────────────────────
statuts = (
    df_s_cmd
    .groupBy("statut")
    .count()
    .orderBy(F.col("count").desc())
    .collect()
)
statut_labels = [r['statut'] for r in statuts]
statut_values = [int(r['count']) for r in statuts]

# ── Ventes par pays ───────────────────────────────────────────
pays_data = (
    df_s_cmd
    .filter(F.col("statut") == "livré")
    .groupBy("pays")
    .agg(F.round(F.sum("montant_total"), 2).alias("ca"),
         F.count("*").alias("nb"))
    .orderBy(F.col("ca").desc())
    .collect()
)
pays_labels = [r['pays'] for r in pays_data]
pays_ca     = [float(r['ca']) for r in pays_data]

print("✅ Données temporelles et géographiques calculées")

✅ Données temporelles et géographiques calculées


In [5]:
# ── Top 10 livres ─────────────────────────────────────────────
top_livres = (
    df_g_top
    .orderBy("rang_ventes")
    .select("titre", "genre", "nb_exemplaires_vendus",
            "chiffre_affaires", "note_moyenne",
            "score_tendance", "performance")
    .limit(10)
    .collect()
)

top_titres  = [r['titre'][:25] + '…' if len(str(r['titre'])) > 25 else str(r['titre']) for r in top_livres]
top_ventes  = [int(r['nb_exemplaires_vendus']) for r in top_livres]
top_scores  = [float(r['score_tendance']) if r['score_tendance'] else 0 for r in top_livres]
top_notes   = [float(r['note_moyenne']) if r['note_moyenne'] else 0 for r in top_livres]

# ── Segments RFM ─────────────────────────────────────────────
rfm_data = (
    df_g_rfm
    .groupBy("segment_rfm")
    .agg(F.count("*").alias("nb"),
         F.round(F.avg("montant_total"), 2).alias("ca_moy"))
    .orderBy(F.col("nb").desc())
    .collect()
)
rfm_labels = [r['segment_rfm'] for r in rfm_data]
rfm_nb     = [int(r['nb']) for r in rfm_data]
rfm_ca     = [float(r['ca_moy']) for r in rfm_data]

# ── Stock par statut ──────────────────────────────────────────
stock_statut = (
    df_s_stock
    .groupBy("statut_stock")
    .count()
    .orderBy(F.col("count").desc())
    .collect()
)
stock_labels = [r['statut_stock'] for r in stock_statut]
stock_values = [int(r['count']) for r in stock_statut]

# ── Recommandations par méthode ───────────────────────────────
reco_methode = (
    df_g_reco
    .groupBy("methode")
    .agg(F.count("*").alias("nb"),
         F.countDistinct("utilisateur_id").alias("nb_users"),
         F.round(F.avg("score_reco"), 3).alias("score_moy"))
    .collect()
)


# ── Tranches horaires ─────────────────────────────────────────
horaires = (
    df_s_cmd
    .groupBy("tranche_horaire")
    .count()
    .orderBy(F.col("count").desc())
    .collect()
)
horaire_labels = [r['tranche_horaire'] for r in horaires]
horaire_values = [int(r['count']) for r in horaires]

# ── Segments utilisateurs Silver ─────────────────────────────
segments = (
    df_s_users
    .groupBy("segment")
    .count()
    .orderBy(F.col("count").desc())
    .collect()
)
seg_labels = [r['segment'] for r in segments]
seg_values = [int(r['count']) for r in segments]

print("✅ Toutes les métriques calculées depuis les vraies données")

✅ Toutes les métriques calculées depuis les vraies données


---
## Étape 3 — Génération du dashboard HTML avec vraies données

In [6]:
# ── Préparer les données JSON pour le dashboard ───────────────
data_json = json.dumps({
    "kpis": {
        "commandes":       kpi_commandes,
        "utilisateurs":    kpi_utilisateurs,
        "livres":          kpi_livres,
        "ca":              float(kpi_ca),
        "panier_moyen":    float(kpi_panier),
        "taux_livraison":  float(kpi_taux_livraison),
    },
    "ca_mensuel":   { "labels": ca_labels,      "values": ca_values,    "cmd": cmd_values },
    "genres":       { "labels": genre_labels,   "ca": genre_ca,         "nb": genre_nb },
    "statuts":      { "labels": statut_labels,  "values": statut_values },
    "pays":         { "labels": pays_labels,    "ca": pays_ca },
    "top_livres":   { "titres": top_titres,     "ventes": top_ventes,   "scores": top_scores, "notes": top_notes },
    "rfm":          { "labels": rfm_labels,     "nb": rfm_nb,           "ca_moy": rfm_ca },
    "stock":        { "labels": stock_labels,   "values": stock_values },
    "horaires":     { "labels": horaire_labels, "values": horaire_values },
    "segments":     { "labels": seg_labels,     "values": seg_values },
    "generated_at": datetime.now().isoformat(),
}, ensure_ascii=False)

print("✅ Données JSON prêtes")
print(f"   Taille JSON : {len(data_json):,} caractères")

✅ Données JSON prêtes
   Taille JSON : 2,316 caractères


In [7]:
# ── Génération du fichier HTML ─────────────────────────────────
html = f"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Smart Library — Dashboard Données Réelles</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
:root {{
  --bg:#ffffff; --bg2:#ffffff; --bg3:#f8fafc; --border:#e2e8f0;
  --a1:#2563eb; --a2:#93c5fd; --a3:#64748b; --a4:#1e293b;
  --a5:#cbd5e1; --a6:#dc2626;
  --t1:#0f172a; --t2:#64748b; --t3:#94a3b8;
  --fh:'Inter',sans-serif; --fb:'Inter',sans-serif; --fm:'Inter',sans-serif;
}}
* {{ margin:0; padding:0; box-sizing:border-box; }}
body {{ background:var(--bg); color:var(--t1); font-family:var(--fb); font-size:14px; -webkit-font-smoothing:antialiased; }}
header {{
  position:sticky; top:0; z-index:100;
  background:var(--bg2);
  border-bottom:1px solid var(--border);
  padding:0 2rem; display:flex; align-items:center;
  justify-content:space-between; height:60px;
}}
.logo {{ display:flex; align-items:center; gap:12px; }}
.logo-icon {{
  width:34px; height:34px; border-radius:8px;
  background:var(--a1);
  display:flex; align-items:center; justify-content:center; font-size:18px;
}}
.logo h1 {{ font-family:var(--fh); font-size:17px; font-weight:700; }}
.logo span {{ color:var(--t2); font-weight:400; }}
.header-right {{ display:flex; align-items:center; gap:16px; }}
.badge {{ padding:3px 10px; border-radius:20px; font-family:var(--fm); font-size:11px; border:1px solid; }}
.b-live {{ background:var(--bg3); border-color:var(--border); color:var(--t2); }}
.b-real {{ background:var(--bg3); border-color:var(--border); color:var(--t2); }}
.gen-date {{ font-family:var(--fm); font-size:11px; color:var(--t3); }}
.tabs {{ display:flex; gap:4px; padding:.8rem 2rem 0; border-bottom:1px solid var(--border); }}
.tab {{ padding:7px 16px; border:none; background:none; color:var(--t2); font-family:var(--fb); font-size:13px; cursor:pointer; border-bottom:2px solid transparent; transition:all .2s; white-space:nowrap; }}
.tab:hover {{ color:var(--t1); }}
.tab.on {{ color:var(--a1); border-bottom-color:var(--a1); }}
main {{ padding:1.5rem 2rem; position:relative; }}
.panel {{ display:none; }}
.panel.on {{ display:block; }}
.kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(165px,1fr)); gap:12px; margin-bottom:1.5rem; }}
.kpi {{ background:var(--bg2); border:1px solid var(--border); border-radius:12px; padding:1.1rem; border-top:2px solid var(--kc,var(--a1)); transition:box-shadow .2s; }}
.kpi:hover {{ box-shadow:0 4px 14px rgba(17,24,39,.06); }}
.kpi-lbl {{ font-size:11px; color:var(--t3); text-transform:uppercase; letter-spacing:.07em; margin-bottom:7px; }}
.kpi-val {{ font-family:var(--fh); font-size:26px; font-weight:800; line-height:1; margin-bottom:5px; }}
.kpi-sub {{ font-size:11px; color:var(--t2); }}
.g2 {{ display:grid; grid-template-columns:1fr 1fr; gap:16px; margin-bottom:1.5rem; }}
.g3 {{ display:grid; grid-template-columns:2fr 1fr; gap:16px; margin-bottom:1.5rem; }}
.card {{ background:var(--bg2); border:1px solid var(--border); border-radius:12px; padding:1.2rem; box-shadow:0 1px 3px rgba(17,24,39,.04); }}
.card-full {{ grid-column:1/-1; }}
.ctitle {{ font-family:var(--fh); font-size:12px; font-weight:600; color:var(--t2); text-transform:uppercase; letter-spacing:.06em; margin-bottom:1rem; display:flex; align-items:center; gap:8px; }}
.dot {{ width:6px; height:6px; border-radius:50%; background:var(--dc,var(--a1)); flex-shrink:0; }}
canvas {{ max-height:260px; }}
.tbl {{ width:100%; border-collapse:collapse; font-size:12px; }}
.tbl th {{ text-align:left; padding:7px 10px; font-size:10px; text-transform:uppercase; letter-spacing:.05em; color:var(--t3); border-bottom:1px solid var(--border); }}
.tbl td {{ padding:9px 10px; border-bottom:1px solid var(--border); color:var(--t1); }}
.tbl tr:last-child td {{ border-bottom:none; }}
.tbl tr:hover td {{ background:rgba(37,99,235,.04); }}
.pill {{ padding:2px 8px; border-radius:20px; font-size:11px; }}
.cow-box {{ background:var(--bg3); border:1px solid rgba(59,130,246,.3); border-radius:10px; padding:1rem; text-align:center; }}
.mor-box {{ background:var(--bg3); border:1px solid rgba(239,68,68,.3); border-radius:10px; padding:1rem; text-align:center; }}
.big-num {{ font-family:var(--fh); font-size:36px; font-weight:800; margin:.3rem 0; }}
.sub-lbl {{ font-size:11px; color:var(--t2); }}
.cow-txt {{ color:var(--a1); }}
.mor-txt {{ color:var(--a6); }}
.vs {{ display:flex; gap:12px; margin-bottom:1rem; }}
.vs > * {{ flex:1; }}
.fact-grid {{ display:grid; grid-template-columns:1fr 1fr; gap:8px; margin-top:.8rem; }}
.fact {{ background:var(--bg3); border:1px solid var(--border); border-radius:8px; padding:.7rem; }}
.fact-lbl {{ font-size:10px; color:var(--t3); text-transform:uppercase; letter-spacing:.05em; margin-bottom:4px; }}
.fact-val {{ font-family:var(--fm); font-size:13px; color:var(--t1); }}
@media(max-width:800px) {{ .g2,.g3 {{ grid-template-columns:1fr; }} }}
</style>
</head>
<body>
<header>
  <div class="logo">
    <div class="logo-icon">📚</div>
    <h1>Smart Library <span>/ Données Réelles</span></h1>
  </div>
  <div class="header-right">
    <span class="badge b-live">● LIVE DATA</span>
    <span class="badge b-real">MinIO + Iceberg</span>
    <span class="gen-date" id="gdate"></span>
  </div>
</header>

<nav class="tabs">
  <button class="tab on" onclick="sw('apercu',this)">Vue d'ensemble</button>
  <button class="tab" onclick="sw('ventes',this)">Ventes</button>
  <button class="tab" onclick="sw('livres',this)">Top Livres</button>
  <button class="tab" onclick="sw('users',this)">Utilisateurs</button>
  <button class="tab" onclick="sw('stock',this)">Stock</button>
</nav>

<main>
<script>const D = {data_json};</script>

<!-- VUE D'ENSEMBLE -->
<div class="panel on" id="p-apercu">
  <div class="kpis" id="kpi-grid"></div>
  <div class="g2">
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a1)"></span>CA mensuel (€)</div><canvas id="cCA"></canvas></div>
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a3)"></span>Ventes par genre</div><canvas id="cGenre"></canvas></div>
  </div>
  <div class="g2">
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a4)"></span>Statuts des commandes</div><canvas id="cStatuts"></canvas></div>
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a5)"></span>CA par pays</div><canvas id="cPays"></canvas></div>
  </div>
</div>

<!-- VENTES -->
<div class="panel" id="p-ventes">
  <div class="card card-full" style="margin-bottom:16px">
    <div class="ctitle"><span class="dot" style="--dc:var(--a1)"></span>Évolution CA mensuel + nb commandes</div>
    <canvas id="cVentes" style="max-height:300px"></canvas>
  </div>
  <div class="g2">
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a2)"></span>Tranches horaires des commandes</div><canvas id="cHoraires"></canvas></div>
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a5)"></span>CA par genre (top 6)</div><canvas id="cGenreCA"></canvas></div>
  </div>
</div>

<!-- TOP LIVRES -->
<div class="panel" id="p-livres">
  <div class="g3" style="margin-bottom:16px">
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a5)"></span>Top 10 — Exemplaires vendus</div><canvas id="cTopVentes"></canvas></div>
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a1)"></span>Score tendance top 10</div><canvas id="cTopScores"></canvas></div>
  </div>
  <div class="card">
    <div class="ctitle"><span class="dot" style="--dc:var(--a5)"></span>Classement complet top 10</div>
    <table class="tbl"><thead><tr><th>#</th><th>Titre</th><th>Ventes</th><th>Note</th><th>Score tendance</th></tr></thead>
    <tbody id="tbl-livres"></tbody></table>
  </div>
</div>

<!-- UTILISATEURS -->
<div class="panel" id="p-users">
  <div class="g2">
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a3)"></span>Segments RFM</div><canvas id="cRFM"></canvas></div>
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a1)"></span>CA moyen par segment RFM</div><canvas id="cRFMca"></canvas></div>
  </div>
  <div class="g2" style="margin-top:16px">
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a4)"></span>Segments Silver (VIP/Régulier/Nouveau)</div><canvas id="cSeg"></canvas></div>
    <div class="card">
      <div class="ctitle"><span class="dot" style="--dc:var(--a5)"></span>Résumé utilisateurs</div>
      <div id="user-facts"></div>
    </div>
  </div>
</div>

<!-- STOCK -->
<div class="panel" id="p-stock">
  <div class="g2">
    <div class="card"><div class="ctitle"><span class="dot" style="--dc:var(--a6)"></span>Stock par statut</div><canvas id="cStock"></canvas></div>
    <div class="card">
      <div class="ctitle"><span class="dot" style="--dc:var(--a4)"></span>Résumé inventaire</div>
      <div id="stock-facts"></div>
    </div>
  </div>
</div>

<!-- HUDI COW vs MOR -->

</main>

<script>
const C = (r,g,b,a=1) => `rgba(${{r}},${{g}},${{b}},${{a}})`;
const PALETTE = [
  C(37,99,235,.9), C(96,165,250,.9), C(148,163,184,.9),
  C(30,41,59,.85), C(203,213,225,.9), C(100,116,139,.6),
  C(37,99,235,.55), C(148,163,184,.5), C(30,41,59,.5), C(203,213,225,.6)
];

const BASE = {{
  responsive:true, maintainAspectRatio:true,
  plugins:{{
    legend:{{ labels:{{ color:'#6b7280', font:{{ family:'Inter', size:11 }}, boxWidth:10 }} }},
    tooltip:{{ backgroundColor:'#111827', borderColor:'#111827', borderWidth:1,
              titleColor:'#ffffff', bodyColor:'#d1d5db', padding:10 }}
  }},
  scales:{{
    x:{{ grid:{{ color:'rgba(17,24,39,.06)' }}, ticks:{{ color:'#9ca3af', font:{{ family:'Inter', size:10 }} }} }},
    y:{{ grid:{{ color:'rgba(17,24,39,.06)' }}, ticks:{{ color:'#9ca3af', font:{{ family:'Inter', size:10 }} }} }}
  }}
}};

function mk(id, type, data, extra={{}}) {{
  const ctx = document.getElementById(id);
  if (!ctx) return;
  return new Chart(ctx, {{ type, data, options:{{ ...BASE, ...extra }} }});
}}

function sw(id, btn) {{
  document.querySelectorAll('.panel').forEach(p => p.classList.remove('on'));
  document.querySelectorAll('.tab').forEach(b => b.classList.remove('on'));
  document.getElementById('p-' + id).classList.add('on');
  btn.classList.add('on');
}}

// ── Init ──────────────────────────────────────────────────────
document.getElementById('gdate').textContent =
  'Généré le ' + new Date(D.generated_at).toLocaleString('fr-FR');

// ── KPIs ──────────────────────────────────────────────────────
const kpiCfg = [
  {{ lbl:'Commandes',    val: D.kpis.commandes.toLocaleString('fr'), sub:'total historique',        c:'var(--a1)' }},
  {{ lbl:'CA livré',     val: Math.floor(D.kpis.ca) + ' €', sub:'hors remboursements',    c:'var(--a4)' }},
  {{ lbl:'Utilisateurs', val: D.kpis.utilisateurs.toLocaleString('fr'), sub:'dans le système',     c:'var(--a2)' }},
  {{ lbl:'Livres',       val: D.kpis.livres.toLocaleString('fr'),    sub:'en catalogue Silver',    c:'var(--a5)' }},
  {{ lbl:'Panier moyen', val: D.kpis.panier_moyen + ' €',            sub:'par commande livrée',    c:'var(--a3)' }},
  {{ lbl:'Taux livraison',val:D.kpis.taux_livraison + '%',           sub:'commandes réussies',     c:'var(--a6)' }},
];
document.getElementById('kpi-grid').innerHTML = kpiCfg.map(k =>
  `<div class="kpi" style="--kc:${{k.c}}">
    <div class="kpi-lbl">${{k.lbl}}</div>
    <div class="kpi-val">${{k.val}}</div>
    <div class="kpi-sub">${{k.sub}}</div>
  </div>`
).join('');

// ── Vue d'ensemble ────────────────────────────────────────────
mk('cCA','bar',{{
  labels: D.ca_mensuel.labels,
  datasets: [{{
    label:'CA (€)', data: D.ca_mensuel.values,
    backgroundColor: C(59,130,246,.7), borderColor: C(59,130,246),
    borderWidth:1, borderRadius:4
  }},{{
    label:'Commandes', data: D.ca_mensuel.cmd,
    type:'line', borderColor: C(6,182,212),
    borderWidth:2, pointRadius:3, fill:false, tension:.4,
    yAxisID:'y2'
  }}]
}},{{
  scales:{{
    x: BASE.scales.x,
    y: {{ ...BASE.scales.y, position:'left' }},
    y2: {{ ...BASE.scales.y, position:'right', grid:{{ drawOnChartArea:false }} }}
  }}
}});

mk('cGenre','doughnut',{{
  labels: D.genres.labels,
  datasets:[{{ data: D.genres.nb, backgroundColor: PALETTE, borderWidth:0, hoverOffset:6 }}]
}},{{ scales:{{}} }});

mk('cStatuts','pie',{{
  labels: D.statuts.labels,
  datasets:[{{ data: D.statuts.values,
    backgroundColor:[C(37,99,235,.85),C(148,163,184,.85),C(220,38,38,.85),C(100,116,139,.6)],
    borderWidth:0, hoverOffset:6 }}]
}},{{ scales:{{}} }});

mk('cPays','bar',{{
  labels: D.pays.labels,
  datasets:[{{ label:'CA (€)', data: D.pays.ca,
    backgroundColor: PALETTE, borderRadius:4, borderWidth:0 }}]
}},{{ plugins:{{ ...BASE.plugins, legend:{{ display:false }} }} }});

// ── Ventes ────────────────────────────────────────────────────
mk('cVentes','bar',{{
  labels: D.ca_mensuel.labels,
  datasets:[{{
    label:'CA (€)', data: D.ca_mensuel.values,
    backgroundColor: C(59,130,246,.6), borderRadius:4, borderWidth:0
  }},{{
    label:'Commandes', data: D.ca_mensuel.cmd,
    type:'line', borderColor: C(100,116,139),
    borderWidth:2, pointRadius:3, fill:false, tension:.4, yAxisID:'y2'
  }}]
}},{{
  scales:{{
    x: BASE.scales.x,
    y:{{ ...BASE.scales.y, position:'left' }},
    y2:{{ ...BASE.scales.y, position:'right', grid:{{ drawOnChartArea:false }} }}
  }}
}});

mk('cHoraires','bar',{{
  labels: D.horaires.labels,
  datasets:[{{ label:'Commandes', data: D.horaires.values,
    backgroundColor: PALETTE, borderRadius:4, borderWidth:0 }}]
}},{{ plugins:{{ ...BASE.plugins, legend:{{ display:false }} }} }});

mk('cGenreCA','bar',{{
  labels: D.genres.labels.slice(0,6),
  datasets:[{{ label:'CA (€)', data: D.genres.ca.slice(0,6),
    backgroundColor: PALETTE, borderRadius:4, borderWidth:0 }}]
}},{{
  indexAxis:'y',
  plugins:{{ ...BASE.plugins, legend:{{ display:false }} }}
}});

// ── Top Livres ────────────────────────────────────────────────
mk('cTopVentes','bar',{{
  labels: D.top_livres.titres,
  datasets:[{{ label:'Exemplaires vendus', data: D.top_livres.ventes,
    backgroundColor: D.top_livres.ventes.map((_,i) =>
      i===0?C(37,99,235,.95):i<3?C(96,165,250,.85):C(203,213,225,.6)
    ), borderRadius:4, borderWidth:0 }}]
}},{{
  indexAxis:'y',
  plugins:{{ ...BASE.plugins, legend:{{ display:false }} }}
}});

mk('cTopScores','bar',{{
  labels: D.top_livres.titres,
  datasets:[{{ label:'Score tendance', data: D.top_livres.scores,
    backgroundColor: C(6,182,212,.8), borderRadius:4, borderWidth:0 }}]
}},{{
  indexAxis:'y',
  plugins:{{ ...BASE.plugins, legend:{{ display:false }} }}
}});

document.getElementById('tbl-livres').innerHTML =
  D.top_livres.titres.map((t,i) => `
  <tr>
    <td style="font-family:var(--fm);color:${{i<3?'var(--a5)':'var(--t3)'}};font-weight:700">${{i+1}}</td>
    <td>${{t}}</td>
    <td>${{D.top_livres.ventes[i]}}</td>
    <td style="color:var(--t1)">${{'★'.repeat(Math.round(D.top_livres.notes[i]))}}${{D.top_livres.notes[i]}}</td>
    <td>
      <div style="display:flex;align-items:center;gap:8px">
        <div style="flex:1;height:4px;background:var(--bg3);border-radius:2px">
          <div style="height:100%;width:${{D.top_livres.scores[i]/10*100}}%;background:linear-gradient(90deg,var(--a1),var(--a2));border-radius:2px"></div>
        </div>
        <span style="font-family:var(--fm);font-size:11px;color:var(--t2)">${{D.top_livres.scores[i]}}</span>
      </div>
    </td>
  </tr>`
).join('');

// ── Utilisateurs ──────────────────────────────────────────────
mk('cRFM','doughnut',{{
  labels: D.rfm.labels,
  datasets:[{{ data: D.rfm.nb, backgroundColor: PALETTE, borderWidth:0, hoverOffset:8 }}]
}},{{ scales:{{}} }});

mk('cRFMca','bar',{{
  labels: D.rfm.labels,
  datasets:[{{ label:'CA moyen (€)', data: D.rfm.ca_moy,
    backgroundColor: PALETTE, borderRadius:4, borderWidth:0 }}]
}},{{ plugins:{{ ...BASE.plugins, legend:{{ display:false }} }} }});

mk('cSeg','doughnut',{{
  labels: D.segments.labels,
  datasets:[{{ data: D.segments.values,
    backgroundColor:[C(30,41,59,.85),C(37,99,235,.85),C(148,163,184,.85)],
    borderWidth:0, hoverOffset:6 }}]
}},{{ scales:{{}} }});

document.getElementById('user-facts').innerHTML = `
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px">
    ${{D.segments.labels.map((l,i) => `
      <div style="background:var(--bg3);border:1px solid var(--border);border-radius:8px;padding:.7rem">
        <div style="font-size:10px;color:var(--t3);text-transform:uppercase;letter-spacing:.05em;margin-bottom:4px">${{l}}</div>
        <div style="font-family:var(--fm);font-size:20px;font-weight:700;color:var(--t1)">${{D.segments.values[i]}}</div>
      </div>`).join('')}}
  </div>`;

// ── Stock ─────────────────────────────────────────────────────
mk('cStock','doughnut',{{
  labels: D.stock.labels,
  datasets:[{{ data: D.stock.values,
    backgroundColor:[C(220,38,38,.85),C(203,213,225,.9),C(37,99,235,.85),C(100,116,139,.6)],
    borderWidth:0, hoverOffset:6 }}]
}},{{ scales:{{}} }});

document.getElementById('stock-facts').innerHTML = `
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:8px">
    ${{D.stock.labels.map((l,i) => `
      <div style="background:var(--bg3);border:1px solid var(--border);border-radius:8px;padding:.7rem">
        <div style="font-size:10px;color:var(--t3);text-transform:uppercase;margin-bottom:4px">${{l}}</div>
        <div style="font-family:var(--fm);font-size:20px;font-weight:700">${{D.stock.values[i]}} livres</div>
      </div>`).join('')}}
  </div>`;



</script>
</body>
</html>
"""

# ── Sauvegarder le dashboard ──────────────────────────────────
output_path = "/home/jovyan/notebooks/dashboard_reel.html"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✅ Dashboard généré : {output_path}")
print(f"   Ouvre-le dans ton navigateur ou depuis JupyterLab")
print(f"   http://localhost:8889/files/notebooks/dashboard_reel.html")

✅ Dashboard généré : /home/jovyan/notebooks/dashboard_reel.html
   Ouvre-le dans ton navigateur ou depuis JupyterLab
   http://localhost:8889/files/notebooks/dashboard_reel.html


In [8]:
spark.stop()
print("✅ SparkSession fermée — Dashboard généré avec succès !")
print()
print("📊 Accède au dashboard ici :")
print("   http://localhost:8889/files/notebooks/dashboard_reel.html")

✅ SparkSession fermée — Dashboard généré avec succès !

📊 Accède au dashboard ici :
   http://localhost:8889/files/notebooks/dashboard_reel.html
